# API Gateway HTTP API with Boto3

Region: ap-south-1

In [ ]:
# create the api gateway, lambda, and iam clients
import boto3
import json
import time
import uuid
import zipfile
import io

REGION = "ap-south-1"
apigw = boto3.client("apigatewayv2", region_name=REGION)
lambda_client = boto3.client("lambda", region_name=REGION)
iam = boto3.client("iam")
sts = boto3.client("sts")

api_name = f"exam-http-api-{uuid.uuid4().hex[:8]}"
stage_name = "prod"
role_name = f"exam-apigw-role-{uuid.uuid4().hex[:8]}"
function_name = f"exam-apigw-fn-{uuid.uuid4().hex[:8]}"
statement_id = f"apigw-invoke-{uuid.uuid4().hex[:8]}"

assume_role_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

lambda_code = [
    "def lambda_handler(event, context):\n",
    "    return {\"statusCode\": 200, \"body\": \"hello from api gateway\"}\n",
]

buffer = io.BytesIO()
with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.writestr("lambda_function.py", "".join(lambda_code))
zip_bytes = buffer.getvalue()

In [ ]:
# create the lambda role and function
response = iam.create_role(
    RoleName=role_name,
    AssumeRolePolicyDocument=json.dumps(assume_role_policy),
)
print(response)

response = iam.attach_role_policy(
    RoleName=role_name,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
)
print(response)

time.sleep(10)

response = lambda_client.create_function(
    FunctionName=function_name,
    Runtime="python3.11",
    Role=iam.get_role(RoleName=role_name)["Role"]["Arn"],
    Handler="lambda_function.lambda_handler",
    Code={"ZipFile": zip_bytes},
    Timeout=10,
)
print(response)

lambda_arn = response["FunctionArn"]

In [ ]:
# create the http api and integration
response = apigw.create_api(
    Name=api_name,
    ProtocolType="HTTP",
)
print(response)

api_id = response["ApiId"]

response = apigw.create_integration(
    ApiId=api_id,
    IntegrationType="AWS_PROXY",
    IntegrationUri=lambda_arn,
    PayloadFormatVersion="2.0",
)
print(response)

integration_id = response["IntegrationId"]

response = apigw.create_route(
    ApiId=api_id,
    RouteKey="GET /",
    Target=f"integrations/{integration_id}",
)
print(response)

route_id = response["RouteId"]

response = apigw.create_stage(
    ApiId=api_id,
    StageName=stage_name,
    AutoDeploy=True,
)
print(response)

In [ ]:
# add permission and print the endpoint
account_id = sts.get_caller_identity()["Account"]

response = lambda_client.add_permission(
    FunctionName=lambda_arn,
    StatementId=statement_id,
    Action="lambda:InvokeFunction",
    Principal="apigateway.amazonaws.com",
    SourceArn=f"arn:aws:execute-api:{REGION}:{account_id}:{api_id}/*/*",
)
print(response)

print(f"api endpoint: https://{api_id}.execute-api.{REGION}.amazonaws.com/{stage_name}/")

In [ ]:
# get the api details
response = apigw.get_api(
    ApiId=api_id,
)
print(response)

In [ ]:
# delete the api, lambda function, and role
response = apigw.delete_route(
    ApiId=api_id,
    RouteId=route_id,
)
print(response)

response = apigw.delete_integration(
    ApiId=api_id,
    IntegrationId=integration_id,
)
print(response)

response = apigw.delete_api(
    ApiId=api_id,
)
print(response)

response = lambda_client.delete_function(
    FunctionName=function_name,
)
print(response)

response = iam.detach_role_policy(
    RoleName=role_name,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
)
print(response)

response = iam.delete_role(
    RoleName=role_name,
)
print(response)